In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('your_file.txt', sep='\t', header=None, names=['contig', 'length', 'evalue', 'identity', 'alignment_length', 'protein', 'description', 'accession'])

df['log_evalue'] = -np.log10(df['evalue'].replace(0, 1e-180))
df['coverage'] = df['alignment_length'] / df['length'] * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].barh(df['contig'], df['identity'], color='steelblue')
axes[0, 0].set_xlabel('Identity (%)')
axes[0, 0].set_title('Identity by Contig')
axes[0, 1].scatter(df['length'], df['identity'], c=df['coverage'], cmap='viridis', s=100, alpha=0.6)
plt.colorbar(axes[0, 1].collections[0], ax=axes[0, 1], label='Coverage (%)')
axes[0, 1].set_xlabel('Contig Length (bp)')
axes[0, 1].set_ylabel('Identity (%)')
axes[0, 1].set_title('Length vs Identity')
axes[1, 0].bar(df['contig'], df['log_evalue'], color='coral')
axes[1, 0].set_xlabel('Contig')
axes[1, 0].set_ylabel('-log10(E-value)')
axes[1, 0].set_title('E-value Significance')
axes[1, 1].hist(df['identity'], bins=20, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Identity (%)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Identity Distribution')
plt.tight_layout()
plt.show()



In [ ]:
fig = px.scatter(df, x='length', y='identity', size='alignment_length', color='coverage', hover_name='contig', log_x=True, title='Contig Analysis')
fig.show()



In [ ]:
virus_names = df['description'].str.extract(r'\[(.*?)\]').fillna('Unknown')[0]
df['virus'] = virus_names
pivot = df.pivot_table(index='contig', columns='virus', values='identity', fill_value=0)
plt.figure(figsize=(12, 8))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Identity (%)'})
plt.title('Contig vs Virus Identity Matrix')
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df, x='length', y='identity', size='coverage', hue='virus', sizes=(50, 400), alpha=0.7, palette='tab10')
plt.xscale('log')
plt.xlabel('Contig Length (bp)')
plt.ylabel('Identity (%)')
plt.title('Bubble Plot: Length vs Identity')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()



In [ ]:
fig = go.Figure(data=[go.Sankey(node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), label=['Sample'] + list(df['virus'].unique()) + list(df['contig']), color='blue'), link=dict(source=[0]*len(df), target=[df['virus'].unique().tolist().index(v)+1 for v in df['virus']], value=df['identity'], color='rgba(0,0,255,0.4)'))])
fig.update_layout(title='Sankey Diagram: Sample to Virus', font_size=10)
fig.show()



In [ ]:
viruses_short = df['virus'].str[:30]
plt.figure(figsize=(12, 6))
df_sorted = df.sort_values('identity')
plt.barh(range(len(df_sorted)), df_sorted['coverage'], color='lightgreen', label='Coverage (%)')
plt.barh(range(len(df_sorted)), df_sorted['identity'], color='steelblue', alpha=0.7, label='Identity (%)')
plt.yticks(range(len(df_sorted)), df_sorted['contig'])
plt.xlabel('Percentage')
plt.title('Identity and Coverage by Contig')
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].violinplot([df[df['virus']==v]['identity'] for v in df['virus'].unique()], positions=range(len(df['virus'].unique())), showmeans=True)
axes[0].set_xticks(range(len(df['virus'].unique())))
axes[0].set_xticklabels(df['virus'].unique(), rotation=45, ha='right')
axes[0].set_ylabel('Identity (%)')
axes[0].set_title('Identity Distribution by Virus')
axes[1].boxplot([df[df['virus']==v]['coverage'] for v in df['virus'].unique()], labels=df['virus'].unique())
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Coverage (%)')
axes[1].set_title('Coverage Distribution by Virus')
axes[2].scatter(df['log_evalue'], df['identity'], c=df['coverage'], cmap='plasma', s=80, alpha=0.6)
plt.colorbar(axes[2].collections[0], ax=axes[2], label='Coverage (%)')
axes[2].set_xlabel('-log10(E-value)')
axes[2].set_ylabel('Identity (%)')
axes[2].set_title('E-value vs Identity')
plt.tight_layout()
plt.show()



In [ ]:
from wordcloud import WordCloud
text = ' '.join(df['virus'].dropna())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Virus Distribution Word Cloud')
plt.show()



In [ ]:
df_grouped = df.groupby('virus').agg({'identity': 'mean', 'coverage': 'mean', 'contig': 'count'}).reset_index()
df_grouped.columns = ['virus', 'avg_identity', 'avg_coverage', 'hit_count']
fig = px.scatter_3d(df_grouped, x='avg_identity', y='avg_coverage', z='hit_count', color='virus', size='hit_count', hover_name='virus', title='3D Virus Comparison')
fig.show()



In [ ]:
import networkx as nx
G = nx.Graph()
for _, row in df.iterrows():
    G.add_edge('Sample', row['virus'], weight=row['identity'])
    G.add_edge(row['virus'], row['contig'], weight=row['coverage'])
pos = nx.spring_layout(G, k=2, iterations=50)
plt.figure(figsize=(14, 10))
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=2000, font_size=8, font_weight='bold', edge_color='gray', width=[G[u][v]['weight']/20 for u,v in G.edges()])
plt.title('Network Graph: Sample-Virus-Contig Relationships')
plt.tight_layout()
plt.show()

df['category'] = pd.cut(df['identity'], bins=[0, 50, 70, 90, 100], labels=['Low (<50%)', 'Medium (50-70%)', 'High (70-90%)', 'Very High (>90%)'])
fig = px.sunburst(df, path=['category', 'virus', 'contig'], values='alignment_length', title='Hierarchical View of Viral Hits', color='identity', color_continuous_scale='RdBu')
fig.show()

fig = make_subplots(rows=2, cols=2, specs=[[{'type': 'bar'}, {'type': 'scatter'}], [{'type': 'heatmap'}, {'type': 'bar'}]])
fig.add_trace(go.Bar(x=df['contig'], y=df['identity'], name='Identity'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['length'], y=df['identity'], mode='markers', marker=dict(size=df['coverage']/5, color=df['log_evalue'], colorscale='Viridis'), text=df['virus']), row=1, col=2)
fig.add_trace(go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index, colorscale='Reds'), row=2, col=1)
fig.add_trace(go.Bar(x=df_grouped['virus'], y=df_grouped['hit_count'], name='Hit Count'), row=2, col=2)
fig.update_layout(height=800, title_text='Multi-Panel Viral Metagenomics Dashboard', showlegend=False)
fig.show()